In [ ]:
import os
import torch
import sys
import random
import re
import soundfile as sf
import numpy as np
import datetime as dt
import noisereduce as nr
import IPython.display as ipd
import torchaudio.functional as F_audio

project_root = "/rds/general/user/ak8224/home/MedEmoji-TTS/matcha-tts"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import matcha
print("✅ Loading Matcha-TTS from:", matcha.__file__)

from matcha.models.matcha_tts import MatchaTTS
from matcha.hifigan.models import Generator as HiFiGAN
from matcha.hifigan.denoiser import Denoiser
from matcha.hifigan.config import v1
from matcha.utils.utils import intersperse
from matcha.text.symbols import symbols
from matcha.text import text_to_sequence

# Support Function

In [ ]:
def init_benchmark_model(model_name: str, device: str = "cuda"):
    """
    Initializes a single TTS model to save VRAM.
    model_name: 'matcha_base', 'vits', 'xtts', 'parler', 'microsoft', or 'bark'
    """
    print(f"🚀 Initializing {model_name}...")
    model_dict = {"name": model_name, "device": device}

    if model_name == "aphavoice_female":
        
        CHECKPOINT_PATH = r"/rds/general/user/ak8224/home/MedEmoji-TTS/matcha-tts/logs/aphasia_v2/checkpoints/checkpoint_epoch=499.ckpt"

        def load_matcha(path, name, vocab_size):
            print(f"📦 Loading {name} (Vocab: {vocab_size})...")
            m = MatchaTTS.load_from_checkpoint(
                path, 
                map_location="cuda", 
                strict=False,
                n_vocab=vocab_size,
                n_spks=2, 
                spk_emb_dim=64,
                weights_only=False
            )
            m.mel_mean = torch.full((1, 80, 1), -1.314475) 
            m.mel_std = torch.full((1, 80, 1), 1.667435)
            m.eval()
            return m

        BASE_CHECKPOINT_PATH = "/rds/general/user/ak8224/home/MedEmoji-TTS/matcha-tts/checkpoints/hifigan_univ_v1" 

        class AttrDict(dict):
            def __init__(self, *args, **kwargs):
                super(AttrDict, self).__init__(*args, **kwargs)
                self.__dict__ = self

        def load_base_vocoder():
            print("Loading Universal Base HiFi-GAN...")
            h = AttrDict(v1)
            device = torch.device("cuda")
            hifigan = HiFiGAN(h).to(device)
    
            state_dict = torch.load(BASE_CHECKPOINT_PATH, map_location=device)
    
            if 'generator' in state_dict:
                hifigan.load_state_dict(state_dict['generator'])
            else:
                hifigan.load_state_dict(state_dict)
        
            hifigan.eval()
            hifigan.remove_weight_norm()
            return hifigan

        model_dict["model"] = load_matcha(CHECKPOINT_PATH, "Multi Model", 205)
        model_dict["vocoder"] = load_base_vocoder()

    elif model_name == "aphavoice_male":
        
        CHECKPOINT_PATH = r"/rds/general/user/ak8224/home/MedEmoji-TTS/matcha-tts/logs/aphasia_v2/checkpoints/checkpoint_epoch=499.ckpt"

        def load_matcha(path, name, vocab_size):
            print(f"📦 Loading {name} (Vocab: {vocab_size})...")
            m = MatchaTTS.load_from_checkpoint(
                path, 
                map_location="cuda", 
                strict=False,
                n_vocab=vocab_size,
                n_spks=2, 
                spk_emb_dim=64,
                weights_only=False
            )
            m.mel_mean = torch.full((1, 80, 1), -1.314475) 
            m.mel_std = torch.full((1, 80, 1), 1.667435)
            m.eval()
            return m

        BASE_CHECKPOINT_PATH = "/rds/general/user/ak8224/home/MedEmoji-TTS/matcha-tts/checkpoints/hifigan_univ_v1" 

        class AttrDict(dict):
            def __init__(self, *args, **kwargs):
                super(AttrDict, self).__init__(*args, **kwargs)
                self.__dict__ = self

        def load_base_vocoder():
            print("Loading Universal Base HiFi-GAN...")
            h = AttrDict(v1)
            device = torch.device("cuda")
            hifigan = HiFiGAN(h).to(device)
    
            state_dict = torch.load(BASE_CHECKPOINT_PATH, map_location=device)
    
            if 'generator' in state_dict:
                hifigan.load_state_dict(state_dict['generator'])
            else:
                hifigan.load_state_dict(state_dict)
        
            hifigan.eval()
            hifigan.remove_weight_norm()
            return hifigan

        model_dict["model"] = load_matcha(CHECKPOINT_PATH, "Multi Model", 205)
        model_dict["vocoder"] = load_base_vocoder()
        
    else:
        raise ValueError(f"Unknown model: {model_name}")

    print(f"✅ {model_name} loaded successfully!")
    return model_dict

In [ ]:
def synthesize_benchmark_audio(model_dict: dict, text: str, output_path: str, **kwargs):
    """
    Routes the text to the correct generation method.
    """
    name = model_dict["name"]
    device = model_dict["device"]
    
    print(f"🎙️ Generating with {name}: '{text[:30]}...'")

    if name == "aphavoice_female":
        from matcha.text import text_to_sequence
        from matcha.utils.utils import intersperse
        import noisereduce as nr
        
        # --- 1. PROCESS TEXT ---
        sequence_output = text_to_sequence(text, ['english_cleaners2'])
        if isinstance(sequence_output, tuple):
            sequence = sequence_output[0]
        else:
            sequence = sequence_output
            
        sequence = intersperse(sequence, 0)
        x = torch.tensor(sequence, dtype=torch.long, device=device).unsqueeze(0)
        x_lengths = torch.tensor([x.size(-1)], dtype=torch.long, device=device)
        
        # --- 2. GET GENERATION KWARGS (with your custom defaults) ---
        speaker_id = kwargs.get("matcha_speaker_id", 0) 
        speaker_tensor = torch.tensor([speaker_id], dtype=torch.long, device=device)
        
        n_timesteps = kwargs.get("n_timesteps", 15)
        temp = kwargs.get("temperature", 0.5)
        length_scale = kwargs.get("length_scale", 1.0)

        latent_shift = kwargs.get("latent_shift", None)
        alpha = kwargs.get("alpha", 0.0)

        # --- 3. SYNTHESISE MEL ---
        with torch.no_grad():
            output = model_dict["model"].synthesise(
                x, 
                x_lengths, 
                n_timesteps=n_timesteps, 
                temperature=temp, 
                spks=speaker_tensor,
                length_scale=length_scale,
                latent_shift=latent_shift,
                alpha=alpha
            )
            mel = output['mel']
            
            # Unpack Vocoder
            if isinstance(model_dict["vocoder"], tuple):
                vocoder_model = model_dict["vocoder"][0]
            else:
                vocoder_model = model_dict["vocoder"]
                
            # --- 4. WAVEFORM CONVERSION & 4D TENSOR FIX ---
            if mel.ndim == 4:
                if mel.shape[1] == mel.shape[2] == 80:
                    mel = mel[:, 0, :, :] 
                else:
                    mel = mel.squeeze(1)
                    
            audio = vocoder_model(mel).squeeze()
            
            # --- 5. PEAK NORMALIZATION ---
            audio = audio / (torch.max(torch.abs(audio)) + 1e-6)
            audio_np = audio.cpu().numpy()
            
            # --- 6. SPECTRAL DENOISING ---
            clean_audio_np = nr.reduce_noise(
                y=audio_np, 
                sr=22050, 
                prop_decrease=0.20, 
                n_fft=1024,
                win_length=1024
            )
        
        # --- 7. EXPORT ---
        sf.write(output_path, clean_audio_np, 22050)

    elif name == "aphavoice_male":
        from matcha.text import text_to_sequence
        from matcha.utils.utils import intersperse
        import noisereduce as nr
        
        # --- 1. PROCESS TEXT ---
        sequence_output = text_to_sequence(text, ['english_cleaners2'])
        if isinstance(sequence_output, tuple):
            sequence = sequence_output[0]
        else:
            sequence = sequence_output

        sequence = intersperse(sequence, 0)
        x = torch.tensor(sequence, dtype=torch.long, device=device).unsqueeze(0)
        x_lengths = torch.tensor([x.size(-1)], dtype=torch.long, device=device)
        
        # --- 2. GET GENERATION KWARGS (with your custom defaults) ---
        speaker_id = kwargs.get("matcha_speaker_id", 1) 
        speaker_tensor = torch.tensor([speaker_id], dtype=torch.long, device=device)
        
        n_timesteps = kwargs.get("n_timesteps", 15)
        temp = kwargs.get("temperature", 0.5)
        length_scale = kwargs.get("length_scale", 1.0)

        latent_shift = kwargs.get("latent_shift", None)
        alpha = kwargs.get("alpha", 0.0)

        # --- 3. SYNTHESISE MEL ---
        with torch.no_grad():
            output = model_dict["model"].synthesise(
                x, 
                x_lengths, 
                n_timesteps=n_timesteps, 
                temperature=temp, 
                spks=speaker_tensor,
                length_scale=length_scale,
                latent_shift=latent_shift,
                alpha=alpha
            )
            mel = output['mel']
            
            # Unpack Vocoder
            if isinstance(model_dict["vocoder"], tuple):
                vocoder_model = model_dict["vocoder"][0]
            else:
                vocoder_model = model_dict["vocoder"]
                
            # --- 4. WAVEFORM CONVERSION & 4D TENSOR FIX ---
            if mel.ndim == 4:
                if mel.shape[1] == mel.shape[2] == 80:
                    mel = mel[:, 0, :, :] 
                else:
                    mel = mel.squeeze(1)
                    
            audio = vocoder_model(mel).squeeze()
            
            # --- 5. PEAK NORMALIZATION ---
            audio = audio / (torch.max(torch.abs(audio)) + 1e-6)
            audio_np = audio.cpu().numpy()
            
            # --- 6. SPECTRAL DENOISING ---
            clean_audio_np = nr.reduce_noise(
                y=audio_np, 
                sr=22050, 
                prop_decrease=0.20, 
                n_fft=1024,
                win_length=1024
            )
        
        # --- 7. EXPORT ---
        sf.write(output_path, clean_audio_np, 22050)

    print(f"💾 Saved to {output_path}")

# Load Model & Vector

In [ ]:
active_model = init_benchmark_model("aphavoice_female", device="cuda")

# Generate Test Voice

In [ ]:
def batch_synthesize_audios(text_file_path, output_folder_name, model_dict, **kwargs):
    """
    Reads a text file line-by-line and synthesizes an audio file for each line.
    Saves all output .wav files into the specified output folder.
    Passes any extra **kwargs (like xtts_ref_wav or parler_prompt) directly to the synthesis function.
    """
    os.makedirs(output_folder_name, exist_ok=True)

    with open(text_file_path, 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f if line.strip()]
        
    print(f"Found {len(lines)} valid scripts in '{text_file_path}'.")
    print(f"Saving outputs to folder: '{output_folder_name}'\n")
    print("-" * 50)

    for i, text_line in enumerate(lines, start=1):
        filename = f"script_{i:02d}.wav"
        output_path = os.path.join(output_folder_name, filename)
        
        print(f"[{i:02d}/{len(lines):02d}] Synthesizing: {filename} ...")

        synthesize_benchmark_audio(
            model_dict=model_dict,
            text=text_line,
            output_path=output_path,
            **kwargs 
        )
        
    print("-" * 50)
    print(f"Success! All {len(lines)} audio files have been saved to '{output_folder_name}'.")

In [ ]:
my_text_file = "/rds/general/user/ak8224/home/MedEmoji-TTS/jupyter/model-validation/unseen_test_script_manipulated.txt"
my_output_folder = "accent_new/male/male_mandarin_wavs"

accentvector = "/rds/general/user/ak8224/home/MedEmoji-TTS/vector/single_vector/male_mandarin_cross_vector.pt"
loaded_accent_tensor = torch.load(accentvector, map_location="cuda", weights_only=False)

batch_synthesize_audios(
    text_file_path=my_text_file,
    output_folder_name=my_output_folder,
    model_dict=active_model,
    latent_shift=loaded_accent_tensor,
    alpha=5.0
)

In [ ]:
my_text_file = "/rds/general/user/ak8224/home/MedEmoji-TTS/jupyter/model-validation/unseen_test_script_manipulated.txt"
my_output_folder = "emotion_new/female/female_happy_wavs"

emotionvector = "/rds/general/user/ak8224/home/MedEmoji-TTS/vector/single_vector/female_happy_cross_vector.pt"
loaded_emotion_tensor = torch.load(emotionvector, map_location="cuda", weights_only=False)

batch_synthesize_audios(
    text_file_path=my_text_file,
    output_folder_name=my_output_folder,
    model_dict=active_model,
    latent_shift=loaded_emotion_tensor,
    alpha=5.0
)

# Extract Latent Shift from Model

In [ ]:
import torch
import os

In [ ]:
def extract_cross_checkpoint_accent(finetuned_ckpt_path, base_ckpt_path, accent_id, output_path):
    print("📥 Loading checkpoints into memory...")
    
    # 1. Load both checkpoints
    ft_checkpoint = torch.load(finetuned_ckpt_path, map_location="cpu", weights_only=False)
    base_checkpoint = torch.load(base_ckpt_path, map_location="cpu", weights_only=False)
    
    # 2. Locate the speaker embedding tables
    ft_spk_key = [k for k in ft_checkpoint["state_dict"].keys() if "spk_emb.weight" in k][0]
    base_spk_key = [k for k in base_checkpoint["state_dict"].keys() if "spk_emb.weight" in k][0]
    
    ft_embeddings = ft_checkpoint["state_dict"][ft_spk_key]
    base_embeddings = base_checkpoint["state_dict"][base_spk_key]
    
    print(f"🎯 Fine-Tuned Embedding Shape: {list(ft_embeddings.shape)}")
    print(f"🎯 Base VCTK Embedding Shape: {list(base_embeddings.shape)}")
    
    # 3. Safety Check
    if ft_embeddings.shape[1] != base_embeddings.shape[1]:
        raise ValueError("❌ Dimensionality mismatch! The models have different embedding sizes.")
        
    # 4. Perform Cross-Checkpoint Latent Arithmetic
    target_accent_vector = ft_embeddings[accent_id]
    pure_base_vector = base_embeddings[2] 
    accent_shift_vector = target_accent_vector - pure_base_vector
    
    # 5. Export for Generation
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    torch.save(accent_shift_vector, output_path)
    
    magnitude = torch.norm(accent_shift_vector).item()
    print("-" * 50)
    print(f"✅ Success! Cross-Checkpoint Vector Extracted.")
    print(f"📉 Vector L2 Norm: {magnitude:.4f}")
    print(f"📁 Saved to: {output_path}")

In [ ]:
FT_CKPT = "/rds/general/user/ak8224/home/MedEmoji-TTS/matcha-tts/logs/accent/checkpoints/checkpoint_epoch=499.ckpt"
BASE_CKPT = "/rds/general/user/ak8224/home/MedEmoji-TTS/matcha-tts/checkpoints/matcha_vctk.ckpt"
TARGET_ID = 11 # Speaker ID
OUTPUT = "/rds/general/user/ak8224/home/MedEmoji-TTS/vector/final2/male_mandarin_cross_vector.pt"

extract_cross_checkpoint_accent(FT_CKPT, BASE_CKPT, TARGET_ID, OUTPUT)

In [ ]:
def extract_cross_checkpoint_emotion(finetuned_ckpt_path, base_ckpt_path, emotion_id, output_path):
    print("📥 Loading checkpoints into memory...")
    
    # 1. Load both checkpoints
    ft_checkpoint = torch.load(finetuned_ckpt_path, map_location="cpu", weights_only=False)
    base_checkpoint = torch.load(base_ckpt_path, map_location="cpu", weights_only=False)
    
    # 2. Locate the speaker embedding tables
    ft_spk_key = [k for k in ft_checkpoint["state_dict"].keys() if "spk_emb.weight" in k][0]
    base_spk_key = [k for k in base_checkpoint["state_dict"].keys() if "spk_emb.weight" in k][0]
    
    ft_embeddings = ft_checkpoint["state_dict"][ft_spk_key]
    base_embeddings = base_checkpoint["state_dict"][base_spk_key]
    
    print(f"🎯 Fine-Tuned Embedding Shape: {list(ft_embeddings.shape)}")
    print(f"🎯 Base VCTK Embedding Shape: {list(base_embeddings.shape)}")
    
    # 3. Safety Check
    if ft_embeddings.shape[1] != base_embeddings.shape[1]:
        raise ValueError("❌ Dimensionality mismatch! The models have different embedding sizes.")
        
    # 4. Perform Cross-Checkpoint Latent Arithmetic
    target_emotion_vector = ft_embeddings[emotion_id]
    pure_base_vector = base_embeddings[0] 
    emotion_shift_vector = target_emotion_vector - pure_base_vector
    
    # 5. Export for Generation
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    torch.save(emotion_shift_vector, output_path)
    
    magnitude = torch.norm(emotion_shift_vector).item()
    print("-" * 50)
    print(f"✅ Success! Cross-Checkpoint Vector Extracted.")
    print(f"📉 Vector L2 Norm: {magnitude:.4f}")
    print(f"📁 Saved to: {output_path}")

In [ ]:
FT_CKPT = "/rds/general/user/ak8224/home/MedEmoji-TTS/matcha-tts/logs/emotion/checkpoints/checkpoint_epoch=499.ckpt"
BASE_CKPT = "/rds/general/user/ak8224/home/MedEmoji-TTS/matcha-tts/checkpoints/matcha_vctk.ckpt"
TARGET_ID = 7 # Speaker ID
OUTPUT = "/rds/general/user/ak8224/home/MedEmoji-TTS/vector/final2/female_surprise_cross_vector.pt"

extract_cross_checkpoint_emotion(FT_CKPT, BASE_CKPT, TARGET_ID, OUTPUT)

# Plot Visualization

## Generate UMAP (Delta Calculation)

In [ ]:
import os
import glob
import torch
import torchaudio
import numpy as np
import pandas as pd
import umap
from tqdm import tqdm
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from speechbrain.inference.speaker import EncoderClassifier

speaker_encoder = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb", 
    run_opts={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    savedir="tmp_speechbrain"
)

In [ ]:
def get_delta_umap_df(base_accent_dir, encoder, base_label):
    """
    Extracts audio, calculates the delta shift, runs UMAP, 
    and returns a DataFrame ready for plotting.
    """
    search_path = os.path.join(base_accent_dir, "**/*.wav")
    wav_files = glob.glob(search_path, recursive=True)
    
    script_dict = {}
    
    # 1. Extraction Loop
    for file in tqdm(wav_files, desc=f"Extracting {base_label}"):
        folder_name = os.path.basename(os.path.dirname(file))
        clean_label = folder_name.replace("_wavs", "").replace("_", " ").title()
        script_name = os.path.basename(file)
        
        signal, fs = torchaudio.load(file)
        if fs != 16000:
            signal = torchaudio.transforms.Resample(fs, 16000)(signal)
            
        with torch.no_grad():
            emb = encoder.encode_batch(signal).squeeze().cpu().numpy()
            
        if script_name not in script_dict:
            script_dict[script_name] = {}
        script_dict[script_name][clean_label] = emb

    # 2. Delta Math
    delta_embeddings = []
    labels = []
    
    for script_name, accents_in_script in script_dict.items():
        if base_label not in accents_in_script:
            continue
            
        base_emb = accents_in_script[base_label]
        
        for accent_label, emb in accents_in_script.items():
            if accent_label == base_label:
                continue # Skip Base
                
            delta_emb = emb - base_emb
            delta_embeddings.append(delta_emb)
            labels.append(accent_label)

    X = np.array(delta_embeddings)
    
    # 3. UMAP Projection
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.3, metric='cosine', random_state=42)
    embedding_2d = reducer.fit_transform(X)
    
    return pd.DataFrame({
        'UMAP Dimension 1': embedding_2d[:, 0],
        'UMAP Dimension 2': embedding_2d[:, 1],
        'Target': labels
    })

In [ ]:
def plot_side_by_side_umaps(df_emotion, df_accent):
    """
    Plots two UMAPs side-by-side with horizontal legends below the charts.
    Uses strict dictionary mapping to ensure colors never change between runs.
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 7))
    
    # --- 1. Color Dictionaries ---
    emotion_palette = {
        "Male Happy": "#e2bb00", "Female Happy": "#e2bb00", "Happy": "#e2bb00",
        "Male Sad": "#533fb1", "Female Sad": "#533fb1", "Sad": "#533fb1",
        "Male Angry": "#ff4e45", "Female Angry": "#ff4e45", "Angry": "#ff4e45",
        "Male Surprise": "#02c092", "Female Surprise": "#02c092", "Surprise": "#02c092"
    }
    
    accent_palette = {
        "Male Vietnamese": "#e2bb00", "Female Vietnamese": "#e2bb00", "Vietnamese": "#e2bb00",
        "Male Korean": "#533fb1", "Female Korean": "#533fb1", "Korean": "#533fb1",
        "Male Indian": "#ff6ceb", "Female Indian": "#ff6ceb", "Indian": "#ff6ceb",
        "Male Mandarin": "#02c092", "Female Mandarin": "#02c092", "Mandarin": "#02c092",
        "Male Spanish": "#ff4e45", "Female Spanish": "#ff4e45", "Spanish": "#ff4e45",
        "Male Arabic": "#8f3522", "Female Arabic": "#8f3522", "Arabic": "#8f3522"
    }

    # --- Plot 1: Emotion ---
    sns.scatterplot(
        data=df_emotion, x='UMAP Dimension 1', y='UMAP Dimension 2', 
        hue='Target', palette=emotion_palette, # Pass the dictionary here!
        s=60, alpha=0.85, edgecolor='w', linewidth=0.5, ax=axes[0]
    )
    axes[0].set_title("Female Emotion Latent Shifts", fontsize=14, fontweight='bold', pad=15)
    axes[0].set_xlabel("UMAP 1", fontsize=11)
    axes[0].set_ylabel("UMAP 2", fontsize=11)
    
    handles1, labels1 = axes[0].get_legend_handles_labels()
    cleaned_labels1 = [label.replace("Male ", "").replace("Female ", "") for label in labels1]
    axes[0].legend(handles1, cleaned_labels1, loc='upper center', bbox_to_anchor=(0.5, -0.15), fontsize=14,
                   ncol=2, frameon=True, title="Emotion Shift", title_fontproperties={'weight':'bold'})

    # --- Plot 2: Accent ---
    sns.scatterplot(
        data=df_accent, x='UMAP Dimension 1', y='UMAP Dimension 2', 
        hue='Target', palette=accent_palette,
        s=60, alpha=0.85, edgecolor='w', linewidth=0.5, ax=axes[1]
    )
    axes[1].set_title("Male Accent Latent Shifts", fontsize=14, fontweight='bold', pad=15)
    axes[1].set_xlabel("UMAP 1", fontsize=11)
    axes[1].set_ylabel("") 
    
    handles2, labels2 = axes[1].get_legend_handles_labels()
    cleaned_labels2 = [label.replace("Male ", "").replace("Female ", "") for label in labels2]
    axes[1].legend(handles2, cleaned_labels2, loc='upper center', bbox_to_anchor=(0.5, -0.15), fontsize=14,
                   ncol=2, frameon=True, title="Accent Shift", title_fontproperties={'weight':'bold'})

    # --- Formatting ---
    sns.despine()
    
    plt.tight_layout(rect=[0, 0.05, 1, 1], w_pad=0.3)
    plt.savefig("combined_umap_latent_shifts.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# 1. Generate the DataFrames
df_female_accent = get_delta_umap_df("/rds/general/user/ak8224/home/MedEmoji-TTS/jupyter/model-validation/raw_accent/female", speaker_encoder, "Female Base")
df_male_accent = get_delta_umap_df("/rds/general/user/ak8224/home/MedEmoji-TTS/jupyter/model-validation/raw_accent/male", speaker_encoder, "Male Base")
df_female_emotion = get_delta_umap_df("/rds/general/user/ak8224/home/MedEmoji-TTS/jupyter/model-validation/raw_emotion/female", speaker_encoder, "Female Base")
df_male_emotion = get_delta_umap_df("/rds/general/user/ak8224/home/MedEmoji-TTS/jupyter/model-validation/raw_emotion/male", speaker_encoder, "Male Base")

In [ ]:
plot_side_by_side_umaps(df_female_emotion, df_male_accent)

In [ ]:
plot_side_by_side_umaps(df_male_emotion, df_female_accent)

In [ ]:
def plot_2x2_master_umaps(df_male_emotion, df_female_emotion, df_male_accent, df_female_accent):
    """
    Top Row: Emotions (Male / Female)
    Bottom Row: Accents (Male / Female)
    Shared global legends are cleanly placed at the bottom.
    """
    # 1. Create a taller canvas to fit two rows
    fig, axes = plt.subplots(2, 2, figsize=(12, 11))
    
    # 2. Deep copy dataframes to clean text prefixes
    df_me = df_male_emotion.copy()
    df_fe = df_female_emotion.copy()
    df_ma = df_male_accent.copy()
    df_fa = df_female_accent.copy()
    
    for df in [df_me, df_fe, df_ma, df_fa]:
        df['Clean Target'] = df['Target'].str.replace("Male ", "").str.replace("Female ", "")

    # 3. Simplified color dictionaries
    emotion_palette = {
        "Happy": "#e2bb00", "Sad": "#533fb1", "Angry": "#ff4e45", "Surprise": "#02c092"
    }
    
    accent_palette = {
        "Vietnamese": "#e2bb00", "Korean": "#533fb1", "Indian": "#ff6ceb",
        "Mandarin": "#02c092", "Spanish": "#ff4e45", "Arabic": "#8f3522"
    }

    # Male Emotion (Top Left)
    sns.scatterplot(
        data=df_me, x='UMAP Dimension 1', y='UMAP Dimension 2', 
        hue='Clean Target', palette=emotion_palette, legend=False,
        s=55, alpha=0.85, edgecolor='w', linewidth=0.5, ax=axes[0, 0]
    )
    axes[0, 0].set_title("Male Emotion Latent Shifts", fontsize=14, fontweight='bold', pad=12)
    axes[0, 0].set_ylabel("UMAP 2", fontsize=11)
    axes[0, 0].set_xlabel("") # Remove inner X label

    # Female Emotion (Top Right)
    sns.scatterplot(
        data=df_fe, x='UMAP Dimension 1', y='UMAP Dimension 2', 
        hue='Clean Target', palette=emotion_palette, legend=False,
        s=55, alpha=0.85, edgecolor='w', linewidth=0.5, ax=axes[0, 1]
    )
    axes[0, 1].set_title("Female Emotion Latent Shifts", fontsize=14, fontweight='bold', pad=12)
    axes[0, 1].set_ylabel("") # Hide inner Y label to maximize plot space
    axes[0, 1].set_xlabel("")

    # Male Accent (Bottom Left)
    sns.scatterplot(
        data=df_ma, x='UMAP Dimension 1', y='UMAP Dimension 2', 
        hue='Clean Target', palette=accent_palette, legend=False,
        s=55, alpha=0.85, edgecolor='w', linewidth=0.5, ax=axes[1, 0]
    )
    axes[1, 0].set_title("Male Accent Latent Shifts", fontsize=14, fontweight='bold', pad=12)
    axes[1, 0].set_xlabel("UMAP 1", fontsize=11)
    axes[1, 0].set_ylabel("UMAP 2", fontsize=11)

    # Female Accent (Bottom Right)
    sns.scatterplot(
        data=df_fa, x='UMAP Dimension 1', y='UMAP Dimension 2', 
        hue='Clean Target', palette=accent_palette, legend=False,
        s=55, alpha=0.85, edgecolor='w', linewidth=0.5, ax=axes[1, 1]
    )
    axes[1, 1].set_title("Female Accent Latent Shifts", fontsize=14, fontweight='bold', pad=12)
    axes[1, 1].set_xlabel("UMAP 1", fontsize=11)
    axes[1, 1].set_ylabel("") # Hide inner Y label

    sns.despine()
    from matplotlib.lines import Line2D
    
    # 1. Create unique legend markers for Emotion
    emotion_handles = [Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=10) 
                       for color in emotion_palette.values()]
    emotion_labels = list(emotion_palette.keys())

    # 2. Create unique legend markers for Accent
    accent_handles = [Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=10) 
                      for color in accent_palette.values()]
    accent_labels = list(accent_palette.keys())

    # 3. Add Shared Emotion Legend (Centered under the left half of the grid)
    fig.legend(emotion_handles, emotion_labels, loc='upper center', bbox_to_anchor=(0.28, 0.04), 
               fontsize=13, ncol=2, frameon=True, title="Emotion Shift", title_fontproperties={'weight':'bold', 'size': 13})

    # 4. Add Shared Accent Legend (Centered under the right half of the grid)
    fig.legend(accent_handles, accent_labels, loc='upper center', bbox_to_anchor=(0.72, 0.04), 
               fontsize=13, ncol=3, frameon=True, title="Accent Shift", title_fontproperties={'weight':'bold', 'size': 13})

    # Adjust layout ensuring room is kept below the panels for the global legends
    plt.tight_layout(rect=[0, 0.05, 1, 1], h_pad=2.2, w_pad=0.4)
    plt.savefig("2x2_master_umap_latent_shifts.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# 2. Plot them side-by-side
plot_2x2_master_umaps(df_male_emotion, df_female_emotion, df_male_accent, df_female_accent)

# Accent Analysis using ASR

In [ ]:
import os
import glob
import torch
import subprocess
import librosa
import numpy as np
from tqdm import tqdm
from transformers import pipeline
from speechbrain.inference.classifiers import EncoderClassifier

In [ ]:
external_repo_path = os.path.expanduser("~/.vox-profile-eval")

if not os.path.exists(external_repo_path):
    print("⏳ Downloading Vox-Profile evaluation tools (safely outside your main repo)...")
    subprocess.run(["git", "clone", "https://github.com/tiantiaf0627/vox-profile-release.git", external_repo_path], check=True)

if external_repo_path not in sys.path:
    sys.path.insert(0, external_repo_path)

from src.model.accent.whisper_accent import WhisperWrapper

In [ ]:
def extract_voxprofile_accent_probability(audio_dir, target_accent, model_id="tiantiaf/whisper-large-v3-narrow-accent"):
    print(f"\n📥 Loading Custom Vox-Profile Model: {model_id}...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Load the model using their custom wrapper
    model = WhisperWrapper.from_pretrained(model_id).to(device)
    model.eval()
    
    # The exact labels the model outputs
    english_accent_list = [ 
        'East Asia', 'English', 'Germanic', 'Irish', 
        'North America', 'Northern Irish', 'Oceania', 
        'Other', 'Romance', 'Scottish', 'Semitic', 'Slavic', 
        'South African', 'Southeast Asia', 'South Asia', 'Welsh' 
    ]
    
    accent_to_whisper_map = {
        "indian": "South Asia",
        "arabic": "Semitic",
        "spanish": "Romance",
        "mandarin": "East Asia",
        "korean": "East Asia",
        "vietnamese": "Southeast Asia",
        "base": "English" 
    }
    
    target_accent = target_accent.lower().strip()
    if target_accent not in accent_to_whisper_map:
        raise ValueError(f"❌ Target accent '{target_accent}' not supported.")
        
    whisper_target_category = accent_to_whisper_map[target_accent]
    target_index = english_accent_list.index(whisper_target_category)
    
    search_path = os.path.join(audio_dir, "*.wav")
    wav_files = glob.glob(search_path)
    
    if not wav_files:
        return f"No .wav files found directly in: {audio_dir}"
        
    print(f"🔍 Found {len(wav_files)} files. Extracting probabilities for '{whisper_target_category}'...")
    
    scores = []
    
    with torch.no_grad():
        for file in tqdm(wav_files, desc="Analyzing clips"):
            speech, sr = librosa.load(file, sr=16000)
            speech_tensor = torch.tensor(speech).unsqueeze(0).to(device)
            logits = model(speech_tensor)
            probabilities = torch.nn.functional.softmax(logits, dim=-1).squeeze()
            target_prob = probabilities[target_index].item()
            scores.append(target_prob)

    # --- FINAL STATISTICS REPORT ---
    print("\n" + "="*60)
    print(f"🏆 ACCENT ACQUISITION REPORT: {target_accent.upper()} 🏆")
    print("="*60)
    
    if len(scores) > 0:
        mean_prob = np.mean(scores)
        std_prob = np.std(scores)
        print(f"📁 Target Folder          : {audio_dir}")
        print(f"🎯 Matched Whisper Category: {whisper_target_category}")
        print(f"📈 Mean Prob (%) : {mean_prob * 100:.2f}%")
        print(f"📈 Std Prob (%) : {std_prob * 100:.2f}%")
        print(f"🔢 Total Audio Files Checked: {len(scores)}")
    print("="*60 + "\n")

In [ ]:
target_folder_spanish = r"/rds/general/user/ak8224/home/MedEmoji-TTS/jupyter/model-validation/accent_new/male/male_mandarin_wavs"
extract_voxprofile_accent_probability(audio_dir=target_folder_spanish, target_accent="mandarin")

# Emotion Analysis using F0 and RMS Energy

In [ ]:
import os
import glob
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm

In [ ]:
def analyze_single_emotion_shift(folder_path):
    """
    Analyzes a single folder of generated aphasic speech to determine
    the average F0 variance and RMS energy of the latent shift.
    """
    print(f"🔍 Analyzing Aphasia Emotion Shift in: {folder_path}")
    
    # Grab all .wav files in the target directory
    wav_files = glob.glob(os.path.join(folder_path, "*.wav"))
    
    if not wav_files:
        return "❌ No .wav files found in this directory."

    results = []
    f0_variances = []
    rms_energies = []

    for file in tqdm(wav_files, desc="Extracting Acoustics"):
        try:
            # 1. Load Audio and trim silence
            y, sr = librosa.load(file, sr=16000)
            y_trimmed, _ = librosa.effects.trim(y, top_db=30)
            
            # 2. Extract RMS Energy (Loudness/Intensity)
            rms = librosa.feature.rms(y=y_trimmed)[0]
            mean_rms = np.mean(rms)
            
            # 3. Extract Pitch (F0) Variance using pYIN
            f0, voiced_flag, voiced_probs = librosa.pyin(
                y_trimmed, 
                fmin=librosa.note_to_hz('C2'), 
                fmax=librosa.note_to_hz('C7')  
            )
            
            # Drop the NaN values (unvoiced consonants/silence)
            f0_voiced = f0[~np.isnan(f0)]
            
            # Calculate Standard Deviation of the pitch
            f0_std = np.std(f0_voiced) if len(f0_voiced) > 0 else 0
                
            f0_variances.append(f0_std)
            rms_energies.append(mean_rms)
            
            results.append({
                "File": os.path.basename(file),
                "RMS Energy": mean_rms,
                "F0 Variance (Hz)": f0_std
            })
            
        except Exception as e:
            continue

    # Calculate the final physical averages for this specific shift
    avg_f0_std = np.mean(f0_variances)
    avg_rms = np.mean(rms_energies)

    print("\n" + "="*55)
    print(f"📊 APHASIA SHIFT METRICS: {os.path.basename(folder_path)}")
    print("-" * 55)
    print(f"📈 Average F0 Variance:   {avg_f0_std:.2f} Hz")
    print(f"🔊 Average RMS Energy:    {avg_rms:.4f}")
    print("="*55 + "\n")

    return pd.DataFrame(results), avg_f0_std, avg_rms

In [ ]:
target_folder_angry = r"/rds/general/user/ak8224/home/MedEmoji-TTS/jupyter/model-validation/emotion_new/female/female_surprise_wavs"
angry_df, angry_f0, angry_rms = analyze_single_emotion_shift(target_folder_angry)